# DRL Assignment 1 - Part 2 (DP)\n
Autonomous Drone Rescue Using Dynamic Programming (Value Iteration)\n
\n
This notebook implements:\n
1. Custom Drone Rescue environment\n
2. Value Iteration (theta = 1e-3)\n
3. Policy visualization\n
4. State-value heatmap analysis\n
5. DP scalability discussion

In [ ]:
from datetime import datetime
import platform

print('Execution Timestamp:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Virtual Machine ID:', platform.node() or 'unknown-host')

In [ ]:
# -----------------------------------------
# Part 2 Helper Functions (DP / Value Iter.)
# -----------------------------------------
# This helper block is intentionally kept inside this notebook
# so submission can be a single self-contained file.

from dataclasses import dataclass
from typing import Dict, List, Optional, Set, Tuple
import random
import time

import numpy as np
import matplotlib.pyplot as plt


Action = int
State = Tuple[int, int, int, int]  # (row, col, battery, rescued_mask)


@dataclass
class DPResult:
    """Stores output artifacts from value iteration."""

    values: Dict[State, float]
    policy: Dict[State, Action]
    iterations: int
    final_delta: float
    runtime_sec: float


class DroneRescueEnv:
    """Custom finite MDP for the drone rescue assignment.

    State contains:
    - drone position (row, col)
    - current battery
    - rescue status bitmask
    """

    # Action map: action_id -> (delta_row, delta_col, printable symbol)
    ACTIONS: Dict[Action, Tuple[int, int, str]] = {
        0: (-1, 0, "U"),
        1: (1, 0, "D"),
        2: (0, -1, "L"),
        3: (0, 1, "R"),
        4: (0, 0, "H"),  # Hover
    }

    def __init__(self, group_id: int, seed: Optional[int] = None) -> None:
        self.group_id = group_id
        self.last_digit = group_id % 10
        self.rng = random.Random(group_id if seed is None else seed)

        # Assignment-driven environment size and counts.
        if self.last_digit <= 4:
            self.rows, self.cols = 5, 5
            self.n_rescue = 2
            self.n_charge = 1
            self.n_danger = 3
            self.n_blocked = 2
            self.wind_prob = 0.20
            self.max_steps = 50
        else:
            self.rows, self.cols = 6, 6
            self.n_rescue = 3
            self.n_charge = 2
            self.n_danger = 4
            self.n_blocked = 3
            self.wind_prob = 0.30
            self.max_steps = 75

        # Assignment rule: battery depends on parity of last digit.
        self.max_battery = 10 if (self.last_digit % 2 == 0) else 15

        # Start is fixed at top-left as required.
        self.start = (0, 0)

        # Cell sets for environment semantics.
        self.blocked: Set[Tuple[int, int]] = set()
        self.danger: Set[Tuple[int, int]] = set()
        self.charging: Set[Tuple[int, int]] = set()
        self.wind: Set[Tuple[int, int]] = set()
        self.rescue_points: List[Tuple[int, int]] = []

        self._build_layout()

        # Map rescue cell -> bit index for bitmask updates.
        self.rescue_index = {pos: i for i, pos in enumerate(self.rescue_points)}

        # Runtime fields for sampled simulation via step().
        self.state: Optional[State] = None
        self.step_count = 0

    def _cell_pool(self) -> List[Tuple[int, int]]:
        """All candidate cells except fixed start cell S."""
        return [
            (r, c)
            for r in range(self.rows)
            for c in range(self.cols)
            if (r, c) != self.start
        ]

    def _sample_cells(self, pool: List[Tuple[int, int]], n: int) -> List[Tuple[int, int]]:
        """Sample n non-overlapping cells and remove them from pool."""
        selected = self.rng.sample(pool, n)
        for cell in selected:
            pool.remove(cell)
        return selected

    def _build_layout(self) -> None:
        """Create deterministic layout from group-based random seed."""
        pool = self._cell_pool()

        self.blocked = set(self._sample_cells(pool, self.n_blocked))
        self.danger = set(self._sample_cells(pool, self.n_danger))
        self.charging = set(self._sample_cells(pool, self.n_charge))
        self.rescue_points = self._sample_cells(pool, self.n_rescue)

        # Add wind zones; moderate count keeps transitions interesting and reachable.
        n_wind = 2 if self.rows == 5 else 3
        self.wind = set(self._sample_cells(pool, n_wind))

    def reset(self) -> State:
        """Reset runtime state for trajectory simulation."""
        self.state = (self.start[0], self.start[1], self.max_battery, 0)
        self.step_count = 0
        return self.state

    def in_bounds(self, r: int, c: int) -> bool:
        """Check if a position is inside grid limits."""
        return 0 <= r < self.rows and 0 <= c < self.cols

    def is_terminal(self, state: State) -> bool:
        """Terminal criteria from assignment.

        Terminal if:
        - battery depleted OR
        - all rescue targets completed
        """
        _, _, battery, mask = state
        all_rescued = mask == (1 << self.n_rescue) - 1
        return battery <= 0 or all_rescued

    def valid_actions(self, state: State) -> List[Action]:
        """Return legal actions for this state.

        Here all 5 actions are available in non-terminal states.
        """
        if self.is_terminal(state):
            return []
        return [0, 1, 2, 3, 4]

    def _attempt_move(self, r: int, c: int, action: Action) -> Tuple[int, int]:
        """Apply movement with wall + blocked-cell handling.

        If movement goes out of bounds or into blocked cell, stay in place.
        """
        dr, dc, _ = self.ACTIONS[action]
        nr, nc = r + dr, c + dc
        if not self.in_bounds(nr, nc):
            return r, c
        if (nr, nc) in self.blocked:
            return r, c
        return nr, nc

    def transition_model(self, state: State, action: Action) -> List[Tuple[float, State, float, bool]]:
        """Compute probabilistic transition model P(s', r | s, a).

        Returns list of tuples: (probability, next_state, reward, done)
        """
        if self.is_terminal(state):
            return [(1.0, state, 0.0, True)]

        r, c, battery, mask = state
        transitions: Dict[Tuple[int, int, int, int], Tuple[float, float, bool]] = {}

        # Wind affects movement actions if the CURRENT cell is wind-zone.
        action_candidates: List[Tuple[float, Action]]
        if (r, c) in self.wind and action in [0, 1, 2, 3]:
            disturbed_prob = self.wind_prob
            intended_prob = 1.0 - disturbed_prob

            # Intended action still happens with remaining probability.
            action_candidates = [(intended_prob, action)]

            # Disturbed movement is uniform among four movement directions.
            for a in [0, 1, 2, 3]:
                action_candidates.append((disturbed_prob / 4.0, a))
        else:
            action_candidates = [(1.0, action)]

        for p_act, actual_action in action_candidates:
            nr, nc = r, c
            next_battery = battery

            # Base movement penalty from assignment reward table.
            reward = -1.0

            if actual_action == 4:
                # Hover behavior is special:
                # - on charging: +2 battery (capped)
                # - elsewhere: battery still decreases by 1
                if (r, c) in self.charging:
                    next_battery = min(self.max_battery, battery + 2)
                else:
                    next_battery = battery - 1
            else:
                # Normal movement action.
                nr, nc = self._attempt_move(r, c, actual_action)
                next_battery = battery - 1

                # Entering charging station refills battery and gives bonus reward.
                if (nr, nc) in self.charging:
                    next_battery = self.max_battery
                    reward += 5.0

            next_mask = mask

            # Danger zone penalty.
            if (nr, nc) in self.danger:
                reward += -10.0

            # Rescue reward is one-time per rescue target.
            if (nr, nc) in self.rescue_index:
                bit = 1 << self.rescue_index[(nr, nc)]
                if (mask & bit) == 0:
                    reward += 20.0
                    next_mask = mask | bit

            done = False

            # Battery exhaustion penalty + terminal.
            if next_battery <= 0:
                reward += -20.0
                next_battery = 0
                done = True

            # Also terminal if all rescues completed.
            if next_mask == (1 << self.n_rescue) - 1:
                done = True

            ns = (nr, nc, next_battery, next_mask)

            # Merge duplicate next states by summing probabilities.
            if ns in transitions:
                old_p, old_r, old_d = transitions[ns]
                combined_r = (old_p * old_r + p_act * reward) / (old_p + p_act)
                transitions[ns] = (old_p + p_act, combined_r, old_d or done)
            else:
                transitions[ns] = (p_act, reward, done)

        return [(p, ns, rwd, done) for ns, (p, rwd, done) in transitions.items()]

    def step(self, action: Action) -> Tuple[State, float, bool, Dict[str, float]]:
        """Sample one transition from current state for rollout simulation."""
        assert self.state is not None, "Call reset() before step()."
        choices = self.transition_model(self.state, action)

        probs = [c[0] for c in choices]
        idx = np.random.choice(len(choices), p=probs)
        _, ns, reward, done = choices[idx]

        self.state = ns
        self.step_count += 1

        # Hard step budget termination from assignment.
        if self.step_count >= self.max_steps:
            done = True

        return ns, reward, done, {}

    def render(self) -> None:
        """Print grid with symbols for quick visual verification."""
        grid = [["F" for _ in range(self.cols)] for _ in range(self.rows)]

        for r, c in self.blocked:
            grid[r][c] = "X"
        for r, c in self.danger:
            grid[r][c] = "D"
        for r, c in self.charging:
            grid[r][c] = "C"
        for r, c in self.wind:
            grid[r][c] = "W"
        for r, c in self.rescue_points:
            grid[r][c] = "R"

        sr, sc = self.start
        grid[sr][sc] = "S"

        # Mark current drone position as A during active simulation.
        if self.state is not None:
            dr, dc, _, _ = self.state
            grid[dr][dc] = "A"

        print("\nGrid Layout:")
        for row in grid:
            print(" ".join(row))

    def enumerate_reachable_states(self) -> List[State]:
        """Enumerate reachable states via graph expansion from initial state."""
        init = (self.start[0], self.start[1], self.max_battery, 0)
        visited: Set[State] = {init}
        frontier = [init]

        while frontier:
            s = frontier.pop()
            if self.is_terminal(s):
                continue

            for a in self.valid_actions(s):
                for _, ns, _, _ in self.transition_model(s, a):
                    if ns not in visited:
                        visited.add(ns)
                        frontier.append(ns)

        return sorted(visited)


def value_iteration(env: DroneRescueEnv, theta: float = 1e-3, gamma: float = 0.99) -> DPResult:
    """Compute optimal value function V* and greedy policy π*.

    Stops when max Bellman residual (delta) drops below theta.
    """
    states = env.enumerate_reachable_states()
    V: Dict[State, float] = {s: 0.0 for s in states}

    t0 = time.perf_counter()
    iterations = 0
    final_delta = 0.0

    while True:
        delta = 0.0
        new_V = V.copy()

        for s in states:
            if env.is_terminal(s):
                continue

            action_values = []
            for a in env.valid_actions(s):
                q_sa = 0.0
                for p, ns, r, done in env.transition_model(s, a):
                    q_sa += p * (r + (0.0 if done else gamma * V[ns]))
                action_values.append(q_sa)

            best = max(action_values)
            new_V[s] = best
            delta = max(delta, abs(new_V[s] - V[s]))

        V = new_V
        iterations += 1
        final_delta = delta

        if delta < theta:
            break

    # Extract greedy policy from converged V.
    policy: Dict[State, Action] = {}
    for s in states:
        if env.is_terminal(s):
            continue

        best_a = None
        best_q = -1e18
        for a in env.valid_actions(s):
            q_sa = 0.0
            for p, ns, r, done in env.transition_model(s, a):
                q_sa += p * (r + (0.0 if done else gamma * V[ns]))
            if q_sa > best_q:
                best_q = q_sa
                best_a = a

        policy[s] = int(best_a)

    runtime = time.perf_counter() - t0
    return DPResult(V, policy, iterations, final_delta, runtime)


def action_symbol(a: Action) -> str:
    """Convert action id to short symbol for policy view."""
    return DroneRescueEnv.ACTIONS[a][2]


def visualize_policy(env: DroneRescueEnv, result: DPResult, output_path: str) -> None:
    """Visualize policy slice for battery=max and rescue-mask=0."""
    mask = 0
    battery = env.max_battery

    grid = np.full((env.rows, env.cols), " ", dtype=object)
    for r in range(env.rows):
        for c in range(env.cols):
            if (r, c) in env.blocked:
                grid[r, c] = "X"
            elif (r, c) in env.danger:
                grid[r, c] = "D"
            elif (r, c) in env.charging:
                grid[r, c] = "C"
            elif (r, c) in env.wind:
                grid[r, c] = "W"
            elif (r, c) in env.rescue_index:
                grid[r, c] = "R"
            else:
                s = (r, c, battery, mask)
                grid[r, c] = action_symbol(result.policy[s]) if s in result.policy else "."

    sr, sc = env.start
    grid[sr, sc] = "S"

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.set_title("Policy Visualization (Battery=max, Rescue mask=0)")
    ax.set_xlim(-0.5, env.cols - 0.5)
    ax.set_ylim(env.rows - 0.5, -0.5)
    ax.set_xticks(range(env.cols))
    ax.set_yticks(range(env.rows))
    ax.grid(True)

    for r in range(env.rows):
        for c in range(env.cols):
            ax.text(c, r, str(grid[r, c]), ha="center", va="center", fontsize=14)

    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()


def plot_value_heatmap(env: DroneRescueEnv, result: DPResult, output_path: str) -> None:
    """Plot V*(s) heatmap for a fixed state slice.

    Slice used:
    - battery = max
    - rescue mask = 0 (none rescued yet)
    - varying only position
    """
    battery = env.max_battery
    mask = 0

    data = np.full((env.rows, env.cols), np.nan, dtype=float)
    for r in range(env.rows):
        for c in range(env.cols):
            s = (r, c, battery, mask)
            if s in result.values and (r, c) not in env.blocked:
                data[r, c] = result.values[s]

    plt.figure(figsize=(8, 6))
    plt.imshow(data, cmap="viridis")
    plt.colorbar(label="V*(s)")
    plt.title("State-Value Heatmap (Battery=max, Rescue mask=0)")
    plt.xticks(range(env.cols))
    plt.yticks(range(env.rows))

    for r in range(env.rows):
        for c in range(env.cols):
            if np.isfinite(data[r, c]):
                plt.text(c, r, f"{data[r, c]:.1f}", ha="center", va="center", color="white", fontsize=8)
            elif (r, c) in env.blocked:
                plt.text(c, r, "X", ha="center", va="center", color="red", fontsize=10)

    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()


def simulate_policy(env: DroneRescueEnv, policy: Dict[State, Action]) -> Tuple[float, int]:
    """Run one rollout using learned greedy policy for sanity checking."""
    s = env.reset()
    total_reward = 0.0
    steps = 0

    while not env.is_terminal(s) and steps < env.max_steps:
        if s not in policy:
            break

        a = policy[s]
        s, r, done, _ = env.step(a)
        total_reward += r
        steps += 1

        if done:
            break

    return total_reward, steps

In [ ]:
# Set your group number here
GROUP_ID = 37

env = DroneRescueEnv(GROUP_ID)

print('=== Environment Summary ===')
print('Group:', env.group_id)
print('Grid size:', f'{env.rows}x{env.cols}')
print('Start position:', env.start)
print('Battery max:', env.max_battery)
print('Wind probability:', env.wind_prob)
print('Step limit:', env.max_steps)
print('Rescue targets:', env.rescue_points)
print('Charging stations:', sorted(env.charging))
print('Danger zones:', sorted(env.danger))
print('Blocked cells:', sorted(env.blocked))
print('Wind zones:', sorted(env.wind))

In [ ]:
env.reset()
env.render()

In [ ]:
# Dynamic Programming: Value Iteration
result = value_iteration(env, theta=1e-3, gamma=0.99)

print('=== Value Iteration Results ===')
print('Reachable states:', len(result.values))
print('Convergence iterations:', result.iterations)
print('Runtime (sec):', round(result.runtime_sec, 4))
print('Final delta/error:', round(result.final_delta, 6))

In [ ]:
# Optional sanity-check rollout of learned policy
rollout_reward, rollout_steps = simulate_policy(env, result.policy)
print('One policy rollout reward:', round(rollout_reward, 2))
print('One policy rollout steps:', rollout_steps)

In [ ]:
# Policy visualization (required)
visualize_policy(env, result, 'dp_policy_visualization.png')
img = plt.imread('dp_policy_visualization.png')
plt.figure(figsize=(9, 6))
plt.imshow(img)
plt.axis('off')
plt.title('Policy Visualization')
plt.show()

In [ ]:
# State-value analysis heatmap (required)
plot_value_heatmap(env, result, 'dp_value_heatmap.png')
img = plt.imread('dp_value_heatmap.png')
plt.figure(figsize=(9, 6))
plt.imshow(img)
plt.axis('off')
plt.title('State-Value Heatmap')
plt.show()

## DP Scalability Discussion\n
Tabular DP faces the curse of dimensionality because state count increases with grid cells, battery levels, and rescue-status combinations. If the grid increases to 10x10 and rescue targets increase, the reachable state space and transition evaluations grow rapidly, making convergence slow and memory-heavy. Dynamic weather adds more stochastic state factors, further increasing complexity. Deep RL can handle this better by learning function approximations for value/policy instead of enumerating all states exactly, which is closer to real-world autonomous drone deployment constraints.

## Notes for Submission\n
- Keep `GROUP_ID` updated to your actual team number before final run.\n
- Run all cells top-to-bottom and export notebook to PDF with outputs.